# 👑 Dream Connect AI - 100% Headless Auto Trainer

이 노트북은 **Dream Connect AI 앱**과 연동되는 완전 자동화 파인튜닝 스크립트입니다.
사용자는 왼쪽 열쇠 아이콘(보안 비밀)에 `HF_TOKEN`만 등록하고 **[런타임] -> [모두 실행]**만 누르면 됩니다!
나머지 모든 설정(모델, 데이터셋 경로)은 허깅페이스에 연동된 설정 파일(`training_config.json`)을 통해 100% 자동 세팅됩니다.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes huggingface_hub

In [ ]:
from huggingface_hub import HfApi, login, hf_hub_download
import json
from google.colab import userdata

# 1. Colab 보안 비밀에서 토큰 가져오기 및 자동 로그인
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# 2. 내 계정명(username) 자동 파악
api = HfApi()
user_info = api.whoami()
username = user_info['name']

# 3. 앱이 몰래 올려둔 설정(config.json) 다운로드 및 파싱
HF_DATASET_ID = f"{username}/SFT_Dataset"
config_path = hf_hub_download(repo_id=HF_DATASET_ID, filename="training_config.json", repo_type="dataset")

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

print("🎯 Auto-Loaded Configuration from Dream Connect AI App:")
print(json.dumps(config, indent=2))

HF_MODEL_ID = config["HF_MODEL_ID"]
UNSLOTH_BASE_MODEL = config["UNSLOTH_BASE_MODEL"]


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# config에서 받아온 베이스 모델로 동적 로드!
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = UNSLOTH_BASE_MODEL,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset

# 자동으로 파악된 데이터셋 ID 사용
dataset = load_dataset(HF_DATASET_ID, split = "train")

alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    texts = []
    # 만능 포맷 감지기 (Conversations vs Flat)
    if "conversations" in examples:
        for convs in examples["conversations"]:
            instruction = convs[0]["content"] if len(convs) > 0 else ""
            response = convs[1]["content"] if len(convs) > 1 else ""
            texts.append(alpaca_prompt.format(instruction, response) + EOS_TOKEN)
    else:
        instructions = examples.get("instruction", [])
        responses = examples.get("output", [])
        for instruction, response in zip(instructions, responses):
            texts.append(alpaca_prompt.format(instruction, response) + EOS_TOKEN)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from trl import SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 테스트용 60 스텝. 실제 훈련시 num_train_epochs = 1 권장
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# config에서 정해준 이름(예: WeonJeong/gemma4-e2b-SFT)으로 푸시!
# 1. 일반 LoRA 어댑터 푸시 (백업용)
model.push_to_hub(HF_MODEL_ID, token=hf_token)
# 2. Ollama 호환 GGUF(q4_k_m) 포맷으로 변환 후 푸시 (Phase 4 인젝터용 핵심 파일!)
model.push_to_hub_gguf(HF_MODEL_ID, tokenizer, quantization_method="q4_k_m", token=hf_token)
